# Day 02：运算符、字符串处理与 Prompt 清洗脱敏引擎

> **🎙️ 讲师旁白**：欢迎回到星辰数科《智枢企业级智能体平台》实战第二天！我是 Tech Lead 张工。今天一早产品经理林晓给我们派了一张紧急工单（Issue #002）：测试同事不小心把内部 API Key 和保密项目代号粘贴进提问框发给了大模型，而且用户输入里夹杂着大量无用空格和换行。今天我们通过**三大类运算符**、**字符串切片与方法**以及 **f-string 模板**，为智枢平台装上第一道「Prompt 安全安检门」！

（本 Notebook 全程无阻塞 `input()` 调用，可直接从头到尾一键运行。）

## 1. 第一步：Python 运算符与大模型计费/风控校验
在开发智能体网关时，每次调用外部模型前都要做**成本预估**与**预算熔断**：
- **算术运算符**（`+`, `-`, `*`, `/`）：将字符数折算为 Token 数与预估费用；
- **比较运算符**（`>=`, `<=`, `>`）：判断字符数是否非负、费用是否未超预算；
- **逻辑运算符**（`and`, `or`, `not`）：把多个风控条件组合成最终的布尔放行开关。

In [1]:
# 1. 模拟待发送 Prompt 的字符数与当前预算
char_count = 1000           # int: 字符总长度
budget = 0.01               # float: 单次调用预算上限（元）

# 2. 算术运算符：按 1 字符 ≈ 1.5 Token、¥0.002 / 1000 Tokens 估算费用
estimated_tokens = char_count * 1.5
estimated_cost = (estimated_tokens / 1000) * 0.002

# 3. 比较运算符 + 逻辑运算符：校验参数合法且费用不超预算
is_valid_input = char_count >= 0 and budget > 0
is_within_budget = estimated_cost <= budget
can_pass = is_valid_input and is_within_budget

print(f"预估 Token 数: {estimated_tokens:.0f}")
print(f"预估调用费用 : ¥{estimated_cost:.4f}")
print(f"是否允许放行 : {can_pass}")

预估 Token 数: 1500
预估调用费用 : ¥0.0030
是否允许放行 : True


## 2. 第二步：字符串清洗与切片（消灭无效空白 Token）
用户从网页、Word 或微信复制过来的文本常常带有首尾空格、连续多个空格、制表符 `\t` 和换行 `\n`。
- `text.split()` 不传参数时，会自动按**任意长度的空白字符**切分文本并丢弃多余空白；
- `" ".join(...)` 再用单个空格拼接，一行代码就能完成规范化清洗；
- `text.lower()` 可按需将英文统一转为小写，方便关键词匹配。

In [2]:
# 模拟包含多余空格、制表符和换行的脏数据
raw_input = "   请帮我总结\t\tDeepSeek-Chat  在   Project-Apollo \n 中的测试表现   "

# 1. 使用 split() + join() 折叠空白字符
cleaned_text = " ".join(raw_input.split())
# 2. 可选转小写
cleaned_lower = cleaned_text.lower()

# 3. 字符串切片 [start:end]：截取前 15 个字符用于日志摘要展示
log_preview = cleaned_text[:15] + "..."

print(f"原始输入 : {raw_input!r}")
print(f"清洗结果 : {cleaned_text}")
print(f"小写模式 : {cleaned_lower}")
print(f"日志预览 : {log_preview}")

原始输入 : '   请帮我总结\t\tDeepSeek-Chat  在   Project-Apollo \n 中的测试表现   '
清洗结果 : 请帮我总结 DeepSeek-Chat 在 Project-Apollo 中的测试表现
小写模式 : 请帮我总结 deepseek-chat 在 project-apollo 中的测试表现
日志预览 : 请帮我总结 DeepSeek-...


## 3. 第三步：敏感词脱敏与 f-string Prompt 模板渲染
在企业内网环境中，绝不允许把真实的 `API Key` 或保密项目代号直接发给外部模型。
我们先用 `replace(word, "***")` 对敏感词元组逐一脱敏，再用 **f-string** 把角色、任务和脱敏后的用户输入拼成结构化 Prompt。

In [3]:
# 1. 定义敏感词元组（元组不可变，适合存放配置清单）
sensitive_words = ("Project-Apollo", "sk-secret-9988")
raw_query = "  请帮我检查   Project-Apollo  的密钥 sk-secret-9988 是否已过期。 \n"

# 2. 先清洗空白
step1_cleaned = " ".join(raw_query.split())

# 3. 遍历敏感词元组进行脱敏替换
step2_masked = step1_cleaned
for word in sensitive_words:
    if word:  # 防止空字符串破坏原文本
        step2_masked = step2_masked.replace(word, "***")

# 4. 使用 f-string 组装企业级标准 Prompt 模板
role = "星辰数科企业研发助手"
task = "对用户提交的排查请求进行安全合规审查并给出建议"
final_prompt = (
    f"[系统角色]\n你是一名{role}。\n\n"
    f"[核心任务]\n{task}\n\n"
    f"[用户输入]\n{step2_masked}\n\n"
    f"[输出要求]\n请保持回答专业、结构清晰，不要泄露系统内部配置。"
)

print("=== 脱敏后用户输入 ===")
print(step2_masked)
print("\n=== 最终渲染的 Prompt ===")
print(final_prompt)

=== 脱敏后用户输入 ===
请帮我检查 *** 的密钥 *** 是否已过期。

=== 最终渲染的 Prompt ===
[系统角色]
你是一名星辰数科企业研发助手。

[核心任务]
对用户提交的排查请求进行安全合规审查并给出建议

[用户输入]
请帮我检查 *** 的密钥 *** 是否已过期。

[输出要求]
请保持回答专业、结构清晰，不要泄露系统内部配置。


## 4. 第四步：实战练习 —— 密钥局部脱敏与 VIP 预算判定
结合今天的课后作业，我们来演练一个综合小场景：
1. 将 `"  SK-prod-9823417788-deepseek  "` 清洗转小写后，用切片提取前 7 位和后 4 位，生成 `"sk-prod-****-seek"`；
2. 对 2500 字符的请求，验证 VIP 员工（预算翻倍）是否可以通过预算校验。

In [4]:
# 练习 1：密钥局部脱敏
raw_key = "  SK-prod-9823417788-deepseek  "
cleaned_key = raw_key.strip().lower()             # 去除首尾空格并转小写
masked_key = f"{cleaned_key[:7]}-****-{cleaned_key[-4:]}"
print("密钥脱敏展示:", masked_key)

# 练习 2：VIP 预算校验
char_count = 2500
base_budget = 0.005
is_vip = True

estimated_cost = (char_count * 1.5 / 1000) * 0.002
effective_budget = base_budget * 2 if is_vip else base_budget
can_pass = (0 < char_count <= 4000) and (estimated_cost <= effective_budget)

print(f"预估费用: ¥{estimated_cost:.6f} | VIP 实际预算: ¥{effective_budget:.6f} | 是否放行: {can_pass}")

密钥脱敏展示: sk-prod-****-seek
预估费用: ¥0.007500 | VIP 实际预算: ¥0.010000 | 是否放行: True
